# Leitor de cédulas para pessoas cegas — pipeline completo

Este notebook reproduz o projeto do zero: baixa os dados, prepara, treina o baseline e a rede, avalia e abre a demo.

**Antes de começar:** `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4)`.

## 1. Código e dependências

In [ ]:
!git clone https://github.com/ifsc-sj-projetos-ia/2026-2-leitura-de-cedulas.git
%cd 2026-2-leitura-de-cedulas
!pip install -q -r requirements.txt

## 2. Dados do Roboflow

Crie a chave em https://app.roboflow.com/settings/api e salve-a nos **Secrets** do Colab (ícone de chave na barra lateral) com o nome `ROBOFLOW_API_KEY`.
Os datasets usados estão em `configs/datasets.yaml`.

In [ ]:
import os
from google.colab import userdata
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
!python -m cedulas.baixar_roboflow

## 3. Fotos próprias (opcional, mas recomendado)

Coloque as fotos no Drive em `MyDrive/cedulas/proprias/<valor>/...` seguindo `data/proprias/README.md`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r /content/drive/MyDrive/cedulas/proprias/* data/proprias/ 2>/dev/null || echo "Sem fotos próprias no Drive"

## 4. Preparação (recorte, deduplicação, split agrupado)

In [ ]:
!python -m cedulas.preparar

## 5. Baseline: histograma de cor + regressão logística

In [ ]:
!python -m cedulas.baseline

## 6. Rede neural: fine-tuning YOLO11-cls

In [ ]:
!python -m cedulas.treinar --modelo yolo11n-cls.pt --epocas 30 --nome yolo11n
!python -m cedulas.treinar --modelo yolo11s-cls.pt --epocas 30 --nome yolo11s

## 7. Avaliação no teste

In [ ]:
!python -m cedulas.avaliar --pesos runs/classify/yolo11n/weights/best.pt --nome yolo11n
!python -m cedulas.avaliar --pesos runs/classify/yolo11s/weights/best.pt --nome yolo11s

In [ ]:
from IPython.display import Image, display
for nome in ["yolo11n", "yolo11s"]:
    display(Image(f"results/{nome}/matriz_confusao.png", width=450))
    display(Image(f"results/{nome}/limiar.png", width=450))

## 8. Experimento: generalização para fotos reais

Treina só com Roboflow e testa nas fotos próprias. A diferença de acurácia entre as fontes mostra o quanto o modelo sofre com a mudança de domínio.

In [ ]:
!python -m cedulas.preparar --proprias-so-teste --saida data/processed_dominio
!python -m cedulas.treinar --dados data/processed_dominio --nome dominio
!python -m cedulas.avaliar --pesos runs/classify/dominio/weights/best.pt --nome dominio --manifest data/processed_dominio/manifest.csv

## 9. Salvar resultados e pesos no Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/cedulas/modelos
!for d in runs/classify/*/; do n=$(basename $d); cp $d/weights/best.pt /content/drive/MyDrive/cedulas/modelos/$n.pt; done
!cp -r results /content/drive/MyDrive/cedulas/

## 10. Demo no celular

Abra o link `https://….gradio.live` que aparecer abaixo no navegador do celular, tire a foto da nota e ouça o valor.

In [ ]:
!python -m cedulas.app --pesos runs/classify/yolo11n/weights/best.pt --compartilhar